# Introduction

We are now moving to the final part of the workshop, which involves formulating business recommendations. Our tasks are:
- Determining a global betting odds,
- Dividing the dataset into categories: A, B, C, D, where A is the best group and D is the weakest group,
- Determining the risk of odds based on accepted parameters for each category.

As the last task, in a discussion format, we must consider the fact that we are a new betting company. When formulating our recommendations, we need to identify the risks that may affect our operations. We will perform this task together in a brainstorming session.

# Notebook Configuration

## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Loading data into the workspace

> Remember to correctly specify the column separator

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("hockey_teams.csv", sep=";")
print(df.head(5))


Saving hockey_teams.csv to hockey_teams.csv
                 team  season  victories  defeats  overtime_defeats  \
0       Boston Bruins    1990         44       24                 0   
1      Buffalo Sabres    1990         31       30                 0   
2      Calgary Flames    1990         46       26                 0   
3  Chicago Blackhawks    1990         49       23                 0   
4   Detroit Red Wings    1990         34       38                 0   

   victory_percentage  scored_goals  received_goals  goal_difference  \
0               64.71           299             264               35   
1               50.82           292             278               14   
2               63.89           344             263               81   
3               68.06           284             211               73   
4               47.22           273             298              -25   

   goals_ratio  
0         1.13  
1         1.05  
2         1.31  
3         1.35  
4         0

### Checking data loading accuracy

In [ ]:
df

,team,season,victories,defeats,overtime_defeats,victory_percentage,scored_goals,received_goals,goal_difference,goals_ratio
0,Boston Bruins,1990,44,24,0,64.71,299,264,35,1.13
1,Buffalo Sabres,1990,31,30,0,50.82,292,278,14,1.05
2,Calgary Flames,1990,46,26,0,63.89,344,263,81,1.31
3,Chicago Blackhawks,1990,49,23,0,68.06,284,211,73,1.35
4,Detroit Red Wings,1990,34,38,0,47.22,273,298,-25,0.92
...,...,...,...,...,...,...,...,...,...,...
577,Tampa Bay Lightning,2011,38,36,8,51.35,235,281,-46,0.84
578,Toronto Maple Leafs,2011,35,37,10,48.61,231,264,-33,0.88
579,Vancouver Canucks,2011,51,22,9,69.86,249,198,51,1.26
580,Washington Capitals,2011,42,32,8,56.76,222,230,-8,0.97


# Determining Betting Odds

Let's review the content of the page: [click](https://trustbet.pl/kursy-bukmacherskie/), where information about methods for determining betting odds can be found. First, we will determine a global odd, which will be the starting point for our analysis (the so-called _baseline scenario_). At this point, we ignore the margin and assume that we are calculating the decimal odd.

Here is the list of steps to be performed to obtain the desired value:
- we will complete the definition of the `get_betting_odds` function, which will take `probability` of a given event as a parameter. We will use it multiple times, so it is worth preparing its implementation now
- then we need to appropriately aggregate the set and determine the **global** probability of the team's victory.

## Implementations of the `get_betting_odds` function

In [ ]:
def get_betting_odds(probability: float):
    return round((1/probability),3)

### Some tests to check the correctness of the implementation

In [ ]:
def test_get_betting_odds():
    assert get_betting_odds(1) == 1, "Expected 1"
    assert get_betting_odds(0.5) == 2, "Expected 2"
    assert get_betting_odds(0.25) == 4, "Expected 4"
    assert get_betting_odds(0.1) == 10, "Expected 10"
    try:
        get_betting_odds(0)
    except ZeroDivisionError:
        pass
    else:
        assert False, "Expected ZeroDivisionError"

    print("All tests passed!")

test_get_betting_odds()

All tests passed!


### Determining the global odds

Here, determine the probability of any team winning

In [ ]:
global_prob  = (df["victory_percentage"].mean())/100

Set the global rate here using the `get_betting_odds` function. Round the result to two decimal places.

In [ ]:
baseline_odd = get_betting_odds(global_prob)
print(baseline_odd)

1.88


# Team Categorization

Let's discuss how we can classify teams into _leagues_. We want to establish 4 leagues:
- A - league consisting of the best teams,
- B - league consisting of good teams,
- C - league consisting of average teams,
- D - league consisting of the weakest teams.

The above terms are quite subjective, so for the purpose of this exercise, we will adopt the following assumptions:
- A - the top 5% of teams,
- B - teams performing better than 70% of the group but worse than league A,
- C - teams performing better than 20% of the group but worse than league B,
- D - the remaining teams.

To accomplish this task, we will additionally implement the function `assign_team_to_league`.

> Note: This task looks unassuming, but it is difficult. Remember that during the class, you have access to the instructor, and later to a mentor.

## Determination of cutoff points for individual leagues

In [ ]:
df_wins_mean = df.groupby("team")["victory_percentage"].mean().sort_values(ascending= False)

In [ ]:
a_league = df_wins_mean.quantile(0.95)
b_league = df_wins_mean.quantile(0.70)
c_league = df_wins_mean.quantile(0.20)

def assign_team_to_league(team: str) -> str:
    value = df_wins_mean.loc[team]
    if value >= a_league:
        return "A league"
    elif value >= b_league:
        return "B league"
    elif value >= c_league:
        return "C league"
    else:
        return "D league"


In [ ]:
print(assign_team_to_league("Detroit Red Wings"))

A league


In [ ]:
df_wins_mean

,victory_percentage
team,
Detroit Red Wings,67.393810
New Jersey Devils,61.521429
Dallas Stars,60.228889
Colorado Avalanche,59.761250
Anaheim Ducks,59.106667
Philadelphia Flyers,58.550952
Boston Bruins,57.010952
St. Louis Blues,56.571905
Pittsburgh Penguins,56.552857


## Determination of odds per league

Here we set the betting odds for each league, which will allow us to draw final conclusions and establish the basic odds for individual teams.

> Remember: After generating the results, it is worth checking if they are reasonable.

In [ ]:
df["league"] = df["team"].apply(assign_team_to_league)


In [ ]:
df_prob = df.groupby("league")["victory_percentage"].mean().round(2).to_frame()

In [ ]:
df_prob["prob"] = df_prob["victory_percentage"]/100
df_prob["odd"] = df_prob["prob"].apply(get_betting_odds)

In [ ]:
df_prob

,victory_percentage,prob,odd
league,,,
A league,64.46,0.6446,1.551
B league,57.37,0.5737,1.743
C league,51.80,0.5180,1.931
D league,43.86,0.4386,2.280


# Discussion

We have obtained certain odds values for each league. But how does this translate into real business? The entire task was about determining certain values from which a bookmaker can begin operations. Correct determination of these values is critical to attract customers to place bets with us, and on the other hand, inappropriate determination may lead to financial losses in the first days of operation.

For this reason, before translating the results and recommendations into business objectives, the analysis is subjected to discussion. Therefore, we will now take on a review role and would like to verify the steps. To that end, we will collectively discuss and critique our work by answering the following questions together:
- What elements of the analysis were simplified? What was omitted in the analysis?
- Are there any inconsistencies in the estimated odds? What are they?
- How can we improve the odds estimates?
- How can we enrich our initial dataset to make the estimates more accurate and less risky?
- How can we simulate the outcomes of our analysis to verify that they do not lead to financial losses?

This is a discussion panel, and every idea is valuable here.


Ignorování domácího/vyjezdního faktoru – v datech není rozlišeno, jestli tým hrál doma nebo venku, přestože to má velký vliv na výsledky.

Chybí síla soupeře – model pracuje jen s průměrnými procenty vítězství, ale nebere v úvahu, proti komu ty výhry byly dosaženy.

Pouze historická data bez trendů – nezkoumáme aktualní situaci výkonu týmu (např. jestli je na vzestupu nebo úpadku).

Žádná informace o soupisce/zraněních atd.

Neřešíme remízy – dataset pracuje jen s výhrami a prohrami.


Stejné váha všech sezon – sezony z 90. let mají stejnou váhu jako poslední sezony.

Nebereme ohled na aktualní týmy co proti sobě hraji.

Dělíme na 4 skupiny a nebereme ohled na to že v každé skupině jsou týmy které mají lepší výkon než ostatní.


Faktor důležitosti – týmy nehrají každý zápas se stejnou motivací (např. zápasy o postup do play-off vs. zápasy “o nic”).

Nezahrnujeme finanční / obchodní stránku – například marže bookmakera nebo limit sázek.

Ignorujeme vliv trhu – naše kurzy nejsou porovnány s reálným trhem, který by mohl ukázat, kde se odchylujeme.

--------------
--------------
How can we simulate the outcomes of our analysis to verify that they do not lead to financial losses?



Můžeme testovat na nějakém ai, vytvoří 1000 sázek a mi spočítáme zda jsme v profitu nebo ne.

Také můžeme testovat podle sebe což není tak efektiví ale může být vice účinný

Backtesting na historických datech – použít naše kurzy na zápasy, kde známe skutečný výsledek, a spočítat, jak by si bookmaker vedl.

Simulace chování zákazníků – testovat, co se stane, když všichni sázejí na oblíbené týmy a naopak.


